In [1]:
# Load extension for running R in Jupyter Notebook
%load_ext rpy2.ipython
# Load extension for autoreloading modules
%load_ext autoreload
%autoreload 2

Error importing in API mode: ImportError("dlopen(/opt/anaconda3/envs/fd_library/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): symbol not found in flat namespace '_R_BaseEnv'")
Trying to import in ABI mode.


In [2]:
import pandas as pd

from utils import (
    functional_richness,
    functional_evenness,
    functional_divergence,
    functional_dispersion,
    raos_Q,
)
from utils import euclidean_distance
from utils import calculate_relative_abundance
from utils import standardize_trait_matrix

## Testing on a small dataset

In [3]:
traits = pd.DataFrame(
    [[1, 2], [2, 3], [3, 1], [4, 2]],
    columns=["Trait_1", "Trait_2"],
    index=["Sp_0", "Sp_1", "Sp_2", "Sp_3"],
)

abundances = pd.DataFrame(
    [[5, 3, 2, 1], [1, 2, 0, 2]],
    columns=["Sp_0", "Sp_1", "Sp_2", "Sp_3"],
    index=["Plot_A", "Plot_B"],
)

### Python usage of the functional diversity functions

In [4]:
FRic = functional_richness(
    abundances, traits, relative_abundance=False, standardize_traits_method="z_score"
)

distance_matrix_euclidean = euclidean_distance(
    traits, metric="euclidean", standardize_method="z_score"
)
FEve = functional_evenness(
    abundances, distance_matrix_euclidean, relative_abundance=False
)

FDiv = functional_divergence(
    abundances, traits, relative_abundance=False, standardize_traits_method="z_score"
)

FDis = functional_dispersion(
    abundances, traits, relative_abundance=False, standardize_traits_method="z_score"
)

raos_Q_df = raos_Q(abundances, distance_matrix_euclidean, relative_abundance=False)

python_results_df = (
    FRic.merge(FEve, on="PID")
    .merge(FDiv, on="PID")
    .merge(FDis, on="PID")
    .merge(raos_Q_df, on="PID")
)
display(python_results_df)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q
0,Plot_A,3.794733,0.734321,0.947928,1.412440,1.68595
1,Plot_B,1.897367,0.989082,0.846568,1.278219,1.63200


In [5]:
r_results_df = None

### R equivalent to calculating the metrics

In [6]:
%%R -i traits,abundances -o r_results_df
library(FD)


trait_mat <- as.matrix(traits)
abun_mat <- as.matrix(abundances)

res <- dbFD(x = trait_mat, a = abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE)

r_results_df <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv,
    R_FDis = res$FDis,
    R_RaoQ = res$RaoQ
)

FRic: No dimensionality reduction was required. The 2 PCoA axes were kept as 'traits'. 


Loading required package: ade4
Loading required package: ape
Loading required package: geometry
Loading required package: vegan
Loading required package: permute


In [8]:
df_merge = python_results_df.merge(r_results_df, on="PID")
df_merge["FRic_ratio"] = df_merge["Functional_Richness"] / df_merge["R_FRic"]
df_merge["FEve_ratio"] = df_merge["Functional_Evenness"] / df_merge["R_FEve"]
df_merge["FDiv_ratio"] = df_merge["Functional_Divergence"] / df_merge["R_FDiv"]
df_merge["FDis_ratio"] = df_merge["Functional_Dispersion"] / df_merge["R_FDis"]
df_merge["RaoQ_ratio"] = df_merge["Raos_Q"] / df_merge["R_RaoQ"]
display(df_merge)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q,R_FRic,R_FEve,R_FDiv,R_FDis,R_RaoQ,FRic_ratio,FEve_ratio,FDiv_ratio,FDis_ratio,RaoQ_ratio
0,Plot_A,3.794733,0.734321,0.947928,1.412440,1.68595,2.846050,0.734321,0.947928,1.063337,1.264463,1.333333,1.0,1.0,1.328309,1.333333
1,Plot_B,1.897367,0.989082,0.846568,1.278219,1.63200,1.423025,0.989082,0.846568,1.090310,1.224000,1.333333,1.0,1.0,1.172345,1.333333


## Testing on the birds dataset

In [9]:
bird_loc = pd.read_csv("./data/example/bird/bird_location.csv")
bird_traits = pd.read_csv("./data/example/bird/bird_traits.csv")

bird_loc = bird_loc.set_index("PID")
bird_traits = bird_traits.set_index("Species")

In [10]:
FRic_bird = functional_richness(
    bird_loc, bird_traits, relative_abundance=False, standardize_traits_method="z_score"
)


distance_matrix_euclidean_bird = euclidean_distance(
    bird_traits, metric="euclidean", standardize_method="z_score"
)
FEve_bird = functional_evenness(
    bird_loc,
    distance_matrix_euclidean_bird,
    relative_abundance=False,
    abundance_weighted=True,
)

FDiv_bird = functional_divergence(
    bird_loc, bird_traits, relative_abundance=False, standardize_traits_method="z_score"
)

FDis_bird = functional_dispersion(
    bird_loc, bird_traits, relative_abundance=False, standardize_traits_method="z_score"
)

distance_matrix_euclidean_bird_z_score = euclidean_distance(
    bird_traits, metric="euclidean", standardize_method="z_score"
)
raos_Q_df_bird = raos_Q(
    bird_loc, distance_matrix_euclidean_bird_z_score, relative_abundance=False
)

python_results_df_bird = (
    FRic_bird.merge(FEve_bird, on="PID")
    .merge(FDiv_bird, on="PID")
    .merge(FDis_bird, on="PID")
    .merge(raos_Q_df_bird, on="PID")
)
display(python_results_df_bird)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q
0,elev_250,66.661794,0.656412,0.747405,1.702557,4.595790
1,elev_500,72.128929,0.651052,0.755107,1.741823,4.719032
2,elev_1000,43.756363,0.623858,0.743327,1.568195,3.962568
3,elev_1500,25.703033,0.568285,0.742684,1.474851,3.603448
4,elev_2000,7.797544,0.605025,0.730325,1.247874,2.375395
5,elev_2500,7.111826,0.631438,0.703676,1.268754,2.521679
6,elev_3000,6.812401,0.616293,0.700852,1.334470,2.691743
7,elev_3500,1.441212,0.592614,0.671813,1.348770,2.938239


In [11]:
r_results_df_bird = None

In [12]:
%%R -i bird_loc,bird_traits -o r_results_df_bird
library(FD)
bird_trait_mat <- as.matrix(bird_traits)
bird_abun_mat <- as.matrix(bird_loc)

res <- dbFD(x = bird_trait_mat, a = bird_abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE, print.pco = TRUE)

r_results_df_bird <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv,
    R_FDis = res$FDis,
    R_RaoQ = res$RaoQ
)

FRic: No dimensionality reduction was required. All 4 PCoA axes were kept as 'traits'. 


In [13]:
merge_df = python_results_df_bird.merge(r_results_df_bird, on="PID")

merge_df["FRic_ratio"] = merge_df["Functional_Richness"] / merge_df["R_FRic"]
merge_df["FEve_ratio"] = merge_df["Functional_Evenness"] / merge_df["R_FEve"]
merge_df["FDiv_ratio"] = merge_df["Functional_Divergence"] / merge_df["R_FDiv"]
merge_df["FDis_ratio"] = merge_df["Functional_Dispersion"] / merge_df["R_FDis"]
merge_df["RaoQ_ratio"] = merge_df["Raos_Q"] / merge_df["R_RaoQ"]

display(merge_df)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q,R_FRic,R_FEve,R_FDiv,R_FDis,R_RaoQ,FRic_ratio,FEve_ratio,FDiv_ratio,FDis_ratio,RaoQ_ratio
0,elev_250,66.661794,0.656412,0.747405,1.702557,4.595790,66.048816,0.656412,0.747405,1.698629,4.574611,1.009281,1.0,1.0,1.002312,1.00463
1,elev_500,72.128929,0.651052,0.755107,1.741823,4.719032,71.465678,0.651052,0.755107,1.737805,4.697285,1.009281,1.0,1.0,1.002312,1.00463
2,elev_1000,43.756363,0.623858,0.743327,1.568195,3.962568,43.354008,0.623858,0.743327,1.564577,3.944307,1.009281,1.0,1.0,1.002312,1.00463
3,elev_1500,25.703033,0.568285,0.742684,1.474851,3.603448,25.466685,0.568285,0.742684,1.471448,3.586842,1.009281,1.0,1.0,1.002312,1.00463
4,elev_2000,7.797544,0.605025,0.730325,1.247874,2.375395,7.725843,0.605025,0.730325,1.244996,2.364448,1.009281,1.0,1.0,1.002312,1.00463
5,elev_2500,7.111826,0.631438,0.703676,1.268754,2.521679,7.046431,0.631438,0.703676,1.265827,2.510058,1.009281,1.0,1.0,1.002312,1.00463
6,elev_3000,6.812401,0.616293,0.700852,1.334470,2.691743,6.749758,0.616293,0.700852,1.331392,2.679338,1.009281,1.0,1.0,1.002312,1.00463
7,elev_3500,1.441212,0.592614,0.671813,1.348770,2.938239,1.427960,0.592614,0.671813,1.345659,2.924699,1.009281,1.0,1.0,1.002312,1.00463


## R vs Python Results

After computing the indices in both R and Python, I compared the results by computing the coresponding ratios. 
FEve and FDiv (metrics bounded between 0-1) seem to be consistent.
FRic, FDis, Rao's Q metrics have discrepencies but they are constant across this example. Could be because of how the FD package computes the distance matrix using Gower's distance then uses PCoA in its calculation

## Tree Dataset

Dataset shape:
    Abundance matrix: 20 Sites x 83 Species
    Traits matrix: 54153 Species x 18 Traits

Pre computing the distance matrix for 54153 species is expensive. Can subset to the 83 present species

In [14]:
tree_loc = pd.read_csv("./data/example/trees/tree_location.csv", index_col=0)
tree_traits = pd.read_csv("./data/example/trees/tree_traits.csv", index_col=0)

print("Tree abundances shape: ", tree_loc.shape)
print("Tree traits shape: ", tree_traits.shape)

Tree abundances shape:  (20, 82)
Tree traits shape:  (54153, 18)


In [15]:
print("Number of Tree species in abundances: ", tree_loc.columns.size)
print("Number of Tree species in traits: ", tree_traits.index.size)

print(
    "Number of common Tree species: ",
    len(tree_loc.columns.intersection(tree_traits.index)),
)

tree_traits_sub = tree_traits.loc[tree_traits.index.intersection(tree_loc.columns)]
print("Shape of subsetted trait matrix: ", tree_traits_sub.shape)

Number of Tree species in abundances:  82
Number of Tree species in traits:  54153
Number of common Tree species:  82
Shape of subsetted trait matrix:  (82, 18)


In [16]:
active_species = tree_loc.sum(axis=0) > 0

cleaned_tree_loc = tree_loc.loc[:, active_species]
cleaned_tree_traits = tree_traits_sub.loc[
    tree_traits_sub.index.intersection(cleaned_tree_loc.columns)
]

print("Original tree_loc shape: ", tree_loc.shape)
print("Cleaned tree_loc shape: ", cleaned_tree_loc.shape)

Original tree_loc shape:  (20, 82)
Cleaned tree_loc shape:  (20, 20)


In [17]:
relative_abundance_tree = calculate_relative_abundance(cleaned_tree_loc)

tree_traits_standardized = standardize_trait_matrix(
    cleaned_tree_traits, method="z_score"
)

distance_matrix_euclidean_tree = euclidean_distance(
    tree_traits_standardized, metric="euclidean", standardize_method=None
)

In [18]:
FRic_tree = functional_richness(
    relative_abundance_tree,
    tree_traits_standardized,
    relative_abundance=True,
    standardize_traits_method=None,
)

FEve_tree = functional_evenness(
    relative_abundance_tree,
    distance_matrix_euclidean_tree,
    relative_abundance=True,
    abundance_weighted=True,
)

FDiv_tree = functional_divergence(
    relative_abundance_tree,
    tree_traits_standardized,
    relative_abundance=True,
    standardize_traits_method=None,
)

FDis_tree = functional_dispersion(
    relative_abundance_tree,
    tree_traits_standardized,
    weighted=True,
    relative_abundance=True,
    standardize_traits_method=None,
)

raos_Q_tree = raos_Q(
    relative_abundance_tree, distance_matrix_euclidean_tree, relative_abundance=True
)

python_results_df_tree = (
    FRic_tree.merge(FEve_tree, on="PID")
    .merge(FDiv_tree, on="PID")
    .merge(FDis_tree, on="PID")
    .merge(raos_Q_tree, on="PID")
)
display(python_results_df_tree)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q
0,2_6_89_24_555,NaN,0.693827,NaN,2.556146,9.164007
1,5_6_19_65993_501,NaN,NaN,NaN,NaN,NaN
2,2_6_49_91_555,NaN,NaN,NaN,0.619684,0.972021
3,5_6_107_54130_501,NaN,NaN,NaN,1.597083,3.985430
4,5_6_39_92314_501,NaN,NaN,NaN,0.105892,0.168245
5,2_6_35_91103_501,NaN,NaN,NaN,NaN,NaN
6,3_6_57_85248_501,NaN,NaN,NaN,1.601909,2.581387
7,3_6_55_52_555,NaN,NaN,NaN,3.623273,14.769124
8,4_6_53_88882_501,NaN,NaN,NaN,NaN,NaN
9,2_6_93_69144_501,NaN,NaN,NaN,NaN,NaN


In [19]:
r_results_df_tree = None

In [20]:
%%R -i cleaned_tree_loc,cleaned_tree_traits -o r_results_df_tree
library(FD)
tree_trait_mat <- as.matrix(cleaned_tree_traits)
tree_abun_mat <- as.matrix(cleaned_tree_loc)

res <- dbFD(x = tree_trait_mat, a = tree_abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE, print.pco = TRUE)

r_results_df_tree <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv,
    R_FDis = res$FDis,
    R_RaoQ = res$RaoQ
)

FEVe: Could not be calculated for communities with <3 functionally singular species. 
FDis: Equals 0 in communities with only one functionally singular species. 
FRic: To respect s > t, FRic could not be calculated for communities with <3 functionally singular species. 
FRic: Dimensionality reduction was required. The last 16 PCoA axes (out of 18 in total) were removed. 
FRic: Quality of the reduced-space representation = 0.6213318 
FDiv: Could not be calculated for communities with <3 functionally singular species. 


In [21]:
merge_df_tree = python_results_df_tree.merge(r_results_df_tree, on="PID")

merge_df_tree["FRic_ratio"] = (
    merge_df_tree["Functional_Richness"] / merge_df_tree["R_FRic"]
)
merge_df_tree["FEve_ratio"] = (
    merge_df_tree["Functional_Evenness"] / merge_df_tree["R_FEve"]
)
merge_df_tree["FDiv_ratio"] = (
    merge_df_tree["Functional_Divergence"] / merge_df_tree["R_FDiv"]
)
merge_df_tree["FDis_ratio"] = (
    merge_df_tree["Functional_Dispersion"] / merge_df_tree["R_FDis"]
)
merge_df_tree["RaoQ_ratio"] = merge_df_tree["Raos_Q"] / merge_df_tree["R_RaoQ"]
display(merge_df_tree)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q,R_FRic,R_FEve,R_FDiv,R_FDis,R_RaoQ,FRic_ratio,FEve_ratio,FDiv_ratio,FDis_ratio,RaoQ_ratio
0,2_6_89_24_555,NaN,0.693827,NaN,2.556146,9.164007,7.524440,0.693827,0.548073,2.491423,8.705807,NaN,1.0,NaN,1.025978,1.052632
1,5_6_19_65993_501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
2,2_6_49_91_555,NaN,NaN,NaN,0.619684,0.972021,NaN,NaN,NaN,0.603993,0.923420,NaN,NaN,NaN,1.025978,1.052632
3,5_6_107_54130_501,NaN,NaN,NaN,1.597083,3.985430,NaN,NaN,NaN,1.556644,3.786158,NaN,NaN,NaN,1.025978,1.052632
4,5_6_39_92314_501,NaN,NaN,NaN,0.105892,0.168245,NaN,NaN,NaN,0.103211,0.159833,NaN,NaN,NaN,1.025978,1.052632
5,2_6_35_91103_501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
6,3_6_57_85248_501,NaN,NaN,NaN,1.601909,2.581387,NaN,NaN,NaN,1.561348,2.452317,NaN,NaN,NaN,1.025978,1.052632
7,3_6_55_52_555,NaN,NaN,NaN,3.623273,14.769124,NaN,NaN,NaN,3.531530,14.030668,NaN,NaN,NaN,1.025978,1.052632
8,4_6_53_88882_501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
9,2_6_93_69144_501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
